In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# PARAMETERS (tune if needed)
# -----------------------------
BLUR_KERNEL = (5, 5)
EDGE_THRESH_LOW = 50
EDGE_THRESH_HIGH = 150
SMOOTHING_WINDOW = 5
IGNORE_TOP_BOTTOM_FRAC = 0.1  # ignore top/bottom 10% of container

# -----------------------------
# LOAD IMAGE
# -----------------------------
image = cv2.imread("1copy2.jpg")
assert image is not None, "Image not found"

orig = image.copy()
h, w = image.shape[:2]

# -----------------------------
# STEP 1: EXTRACT BLUE CHANNEL
# -----------------------------
# Orange juice absorbs blue light strongly → high contrast
blue = image[:, :, 0]

# Reduce noise
blue_blur = cv2.GaussianBlur(blue, BLUR_KERNEL, 0)

# -----------------------------
# STEP 2: FIND CONTAINER REGION
# -----------------------------
# Edge detection
edges = cv2.Canny(blue_blur, EDGE_THRESH_LOW, EDGE_THRESH_HIGH)

# Find contours
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL,
                               cv2.CHAIN_APPROX_SIMPLE)

# Assume largest vertical contour is the container
container_contour = max(contours, key=cv2.contourArea)

# Create mask of container interior
container_mask = np.zeros((h, w), dtype=np.uint8)
cv2.drawContours(container_mask, [container_contour], -1, 255, thickness=-1)

# Bounding box of container
x, y, bw, bh = cv2.boundingRect(container_contour)

# Crop to container region
container_blue = blue_blur[y:y+bh, x:x+bw]
container_mask = container_mask[y:y+bh, x:x+bw]

# -----------------------------
# STEP 3: ROW-WISE INTENSITY SCAN
# -----------------------------
row_means = []
row_stds = []

for row in range(bh):
    mask_row = container_mask[row] > 0
    if np.sum(mask_row) < 10:
        row_means.append(0)
        row_stds.append(0)
        continue

    values = container_blue[row][mask_row]
    row_means.append(np.mean(values))
    row_stds.append(np.std(values))

row_means = np.array(row_means)
row_stds = np.array(row_stds)

# -----------------------------
# STEP 4: FIND LIQUID SURFACE
# -----------------------------
# Compute vertical gradient of intensity
gradient = np.abs(np.gradient(row_means))

# Ignore top/bottom regions (rim + base)
ignore = int(IGNORE_TOP_BOTTOM_FRAC * bh)
valid_range = slice(ignore, bh - ignore)

# Combined score (edge strength - texture penalty)
lambda_texture = 0.3
score = gradient - lambda_texture * row_stds

# Smooth score to suppress pulp noise
score_smooth = np.convolve(
    score, np.ones(SMOOTHING_WINDOW) / SMOOTHING_WINDOW, mode="same"
)

# Liquid surface = strongest horizontal transition
liquid_row = np.argmax(score_smooth[valid_range]) + ignore

# -----------------------------
# STEP 5: COMPUTE FILL RATIO
# -----------------------------
container_top = ignore
container_bottom = bh - ignore

liquid_height_px = container_bottom - liquid_row
container_height_px = container_bottom - container_top

fill_ratio = liquid_height_px / container_height_px

# -----------------------------
# VISUALIZATION
# -----------------------------
vis = orig.copy()
cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 2)
cv2.line(vis, (x, y + liquid_row),
         (x + bw, y + liquid_row), (0, 0, 255), 2)

cv2.putText(
    vis,
    f"Fill: {fill_ratio*100:.1f}%",
    (x, y - 10),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.8,
    (0, 0, 255),
    2,
)

# Show result
cv2.imshow("Orange Juice Fill Detection", vis)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Optional: debug plot
plt.figure(figsize=(6, 4))
plt.plot(row_means, label="Mean Blue Intensity")
plt.plot(score_smooth, label="Detection Score")
plt.axvline(liquid_row, color="r", linestyle="--", label="Detected Surface")
plt.legend()
plt.xlabel("Vertical Pixel Row")
plt.ylabel("Value")
plt.title("Row-wise Intensity Analysis")
plt.show()
